# End-to-End Production Evidence

This notebook reads committed production outputs without rerunning local models. A live request builds providers for that run only, parses PDFs before retrieval when needed, and writes one `AnalysisResult` plus a structured session log.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "integration":
    INTEGRATION = ROOT
else:
    INTEGRATION = ROOT / "integration"

RESULT = INTEGRATION / "results/demo/pdf_ollama_related/analysis_result.json"
TRACE = INTEGRATION / "results/traces/pdf_drq_ollama_related_20260613.jsonl"

In [ ]:
analysis = json.loads(RESULT.read_text(encoding="utf-8"))
{
    "title": analysis["metadata"]["title"],
    "summary": analysis["summary"],
    "recommended_papers": len(analysis["recommended_papers"]),
    "citations": len(analysis["apa_citations"]),
    "provider_sources": analysis["flags"]["provider_sources"],
}

In [ ]:
events = [json.loads(line) for line in TRACE.read_text(encoding="utf-8").splitlines() if line.strip()]
pd.DataFrame(events)[["timestamp", "event", "component", "phase", "status", "source", "duration_ms"]]

In [ ]:
failed = [event for event in events if event.get("status") == "failed"]
assert not failed, failed
assert "[redacted-upload]" not in TRACE.read_text(encoding="utf-8") or all(
    "/private/" not in json.dumps(event) for event in events
)
"Production trace has no failed events and exposes no temporary upload path."

## Live reproduction

From the repository root:

```text
python rpa.py analyze-pdf tests/papers/drq_v2/2107.09645v1.pdf
```

Runtime outputs under `integration/outputs/` and `integration/data/sessions/` are ephemeral. Copy only redacted, reviewed evidence into `integration/results/` for submission.